In [1]:
import sys
import math
from pathlib import Path
!{sys.executable} -m pip install svgwrite ipywidgets

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "aitk" / "robots" / "world.py").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "aitk" / "algorithms"))

for name in list(sys.modules):
    if (
        name == "aitk"
        or name.startswith("aitk.")
        or name == "neat"
        or name.startswith("neat.")
    ):
        del sys.modules[name]

import aitk.robots
import aitk.utils
from aitk.robots.world import World
from aitk.utils import Grid
import ipywidgets as widgets
import neat

widgets.IntSlider()
print("Using aitk from:", aitk.__file__)
print("Using neat from:", neat.__file__)

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Using aitk from: /Users/reishirokawakami/Desktop/Projects/aitk_soccer_experiments/aitk/__init__.py
Using neat from: /Users/reishirokawakami/Desktop/Projects/aitk_soccer_experiments/aitk/algorithms/neat/__init__.py


In [2]:
"""
The setup of the game:

the player will:
- always perceive the goal, the ball, the distance between the goal & the ball
- caluclate the angle and the impact of the hit (= distance)

the ball will:
- once hit, receive the angle and the impact (= distance)
- gradually move towards the angle 
- bounce back if it hits wall 

Evolution:
- individuals: 
    - neural networks that takes in the perceived information and outputs the inputs to the .move() method
- fitness function
    - how quickly the player reaches the ball
    - how close the shot is to the goal 
- NEAT: 
    - evolve both the weights and the structure of the neural networks
"""

def playSoccer(robot):
    """
    Controller function for player robots. 
    If it perceives a nearby ball, calculate the angle and the distance, and move the ball. 
    """

    if "stall_counter" not in robot.state:
        robot.state["stall_counter"] = 0

    if robot.stalled:
        robot.state["stall_counter"] += 1
    else:
        robot.state["stall_counter"] = 0

    if robot.state["stall_counter"] > 10:
        robot.speak("stuck")
        return True

    if robot.kick() == True: 
        ## record the time it takes to first reach the ball
        if robot.discovery_time == math.inf:
            robot.discovery_time = robot.world.time
            print("discovered ball at:", robot.discovery_time)

        robot.speak("kick")
        _, _, degrees = robot.get_pose() # direction it's facing 
        robot_velocity = robot.vx # robot.vy isn't used for .move(), so it stays 0
        robot.stop()

        for ball in test_world._balls:
            ball.impact_from_robot(robot, degrees, robot_velocity)

    for ball in test_world._balls:
        if ball.goal == True:
            robot.speak("goal!")
            robot.goal_time = robot.world.time
            print("goal at:", robot.goal_time)
            return True ## END SIMULATION
        else:
            # robot.move(5, 0)
            robot.move(robot.state["translate"], robot.state["rotate"])
    
    _, _, degrees = robot.get_pose()

    ## FITNESS FUNCTION: we want to evolve the inputs of .move() 




In [3]:
print("Starting...")
test_world = aitk.robots.World(width=200, height=200, scale=2.0)
test_world.add_wall("blue", 181,140,179,60)
robot = aitk.robots.Scribbler(x=50, y=50, a=90, max_trace_length=60)

test_world.add_robot(robot)
test_world.add_ball(100, 75)

robot.state["translate"], robot.state["rotate"] = 5, 0
max_time = 15
test_world.quiet = True
test_world.save()

test_world.watch()
test_world.run([playSoccer])

Starting...
Random seed set to: 170365


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x0…

0it [00:00, ?it/s]

True

## Evolving PlaySoccer()

The code above will help you understand and experiment with the basic setup. From the cells below, we will use NEAT to evolve find soccer player solution.

For the sake of simplicity, we will not make a new BallSensor or GoalSensor objects.

In [4]:
import random

def reset_soccer_episode():
    robot.world.reset()

    ball = robot.world._balls[0]
    ball.x = random.randint(40, 160)
    ball.y = random.randint(40, 160)
    ball.vx = 0
    ball.vy = 0
    ball.goal = False

    robot.x = 75
    robot.y = 75
    robot.a = 0

In [5]:
def eval_fitness(net, max_time, watch=False):
    stall_counter = 0
    reset_soccer_episode()
    robot.state["translate"] = 0
    robot.state["rotate"] = 0
    for i in range(max_time):
        x, y, direction = robot.get_pose()
        if robot.stalled:
            stall_counter += 1
        else:
            stall_counter = 0
        if stall_counter > 10:
            # end evaluation of stalled controllers
            break
        # robot.state["grid"].update(x, y)
        # sensor_vals = get_sensors(robot, max_time)
        # the robot learns the ball/goal distance and angle 
        sensor_vals = robot.update_sensors() ## TODO
        output = net.activate(sensor_vals)
        robot.state["translate"] = output[0]
        robot.state["rotate"] = output[1]
        # Use the same action command for half a second
        test_world.steps(5,[playSoccer], real_time=False, show=watch, quiet=True)

    ## calculate the fitness of the network by time to reach the ball and time to kick the ball into the goal
    fitness = 0

    # reward reaching/kicking the ball earlier
    if robot.discovery_time is not math.inf:
        fitness += 1 - robot.discovery_time / max_time

    # reward scoring earlier, more strongly
    if robot.goal_time is not math.inf:
        fitness += 2 * (1 - robot.goal_time / max_time)
    else:
        fitness -= 1

    return fitness

In [6]:
def eval_population(pop, config):
    """This function must take these two parameters"""
    for genome_id, genome in pop:
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        genome.fitness = eval_fitness(net, 60, watch=False)

In [7]:
def run(config_file):
    # Load configuration
    config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                         neat.DefaultSpeciesSet, neat.DefaultStagnation,
                         config_file)

    # Create the population, which is the top-level object for a NEAT run.
    p = neat.Population(config)

    # Add a stdout reporter to show progress in the terminal.
    p.add_reporter(neat.StdOutReporter(True))
    stats = neat.StatisticsReporter()
    p.add_reporter(stats)
    # Uncomment this to save checkpoints every 5 generations
    # p.add_reporter(neat.Checkpointer(5))

    winner = p.run(eval_population, 10)

    # Display the winning genome.
    print('\nBest genome:\n{!s}'.format(winner))
    node_names = {-1:'d1', -2:'d2', -3:'d3', -4:'timer', 
                  -5:'stall', -6:'bias', 
                  0:'translate', 1: 'rotate'}
    # neat.visualize.draw_net(config, winner, True, node_names=node_names)
    # neat.visualize.plot_stats(stats, ylog=False, view=True)
    # neat.visualize.plot_species(stats, view=True)
    
    return winner

In [8]:
%%writefile coverage.config
#--- parameters for the Robot Coverage experiment ---#

[NEAT]
fitness_criterion     = max
fitness_threshold     = 2.0
pop_size              = 50
reset_on_extinction   = False

[DefaultGenome]
# node activation options
activation_default      = tanh
activation_mutate_rate  = 0.0
activation_options      = tanh

# node aggregation options
aggregation_default     = sum
aggregation_mutate_rate = 0.0
aggregation_options     = sum

# node bias options
bias_init_mean          = 0.0
bias_init_stdev         = 1.0
bias_max_value          = 30.0
bias_min_value          = -30.0
bias_mutate_power       = 0.5
bias_mutate_rate        = 0.7
bias_replace_rate       = 0.1

# genome compatibility options
compatibility_disjoint_coefficient = 1.0
compatibility_weight_coefficient   = 0.5

# connection add/remove rates
conn_add_prob           = 0.5
conn_delete_prob        = 0.5

# connection enable options
enabled_default         = True
enabled_mutate_rate     = 0.01

feed_forward            = True
initial_connection      = full

# node add/remove rates
node_add_prob           = 0.35
node_delete_prob        = 0.35

# network parameters
num_hidden              = 0
num_inputs              = 4
num_outputs             = 2

# node response options
response_init_mean      = 1.0
response_init_stdev     = 0.0
response_max_value      = 30.0
response_min_value      = -30.0
response_mutate_power   = 0.0
response_mutate_rate    = 0.0
response_replace_rate   = 0.0

# connection weight options
weight_init_mean        = 0.0
weight_init_stdev       = 1.0
weight_max_value        = 30
weight_min_value        = -30
weight_mutate_power     = 0.5
weight_mutate_rate      = 0.8
weight_replace_rate     = 0.1

[DefaultSpeciesSet]
compatibility_threshold = 2.0

[DefaultStagnation]
species_fitness_func = max
max_stagnation       = 20
species_elitism      = 1

[DefaultReproduction]
elitism            = 2
survival_threshold = 0.2

Overwriting coverage.config


In [9]:
winner = run("coverage.config")


 ****** Running generation 0 ****** 

discovered ball at: 17.7
discovered ball at: 18.2
discovered ball at: 17.3
discovered ball at: 3.4
discovered ball at: 8.5
discovered ball at: 2.9
Population's average fitness: -0.90267 stdev: 0.26636
Best fitness: -0.04833 - size: (2, 8) - species 2 - id 47
Average adjusted fitness: 0.095
Mean genetic distance 1.434, standard deviation 0.322
Population of 50 members in 2 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    0    26     -0.1    0.107     0
     2    0    24     -0.0    0.083     0
Total extinctions: 0
Generation time: 2.028 sec

 ****** Running generation 1 ****** 

discovered ball at: 3.4
discovered ball at: 8.5
discovered ball at: 5.4
discovered ball at: 29.5
discovered ball at: 9.1
discovered ball at: 2.9
discovered ball at: 17.3
discovered ball at: 22.8
discovered ball at: 12.2
discovered ball at: 15.1
discovered ball at: 11.6
Population's average fitness: -0.82593 stdev: 0.3334

In [10]:
# Test the winning network
config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                         neat.DefaultSpeciesSet, neat.DefaultStagnation,
                         "coverage.config")
winner_net = neat.nn.FeedForwardNetwork.create(winner, config)
eval_fitness(winner_net, 60, watch=True)

discovered ball at: 4.4
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0
goal at: 11.0


2.56